In [4]:
%%capture
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
!pip install lime
from lime.lime_tabular import LimeTabularExplainer
from lime import submodular_pick
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from collections import defaultdict
from sklearn.metrics import accuracy_score
import shap
import warnings
warnings.filterwarnings('ignore')

# XAI Lab: LIME and SHAP

In [2]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

display(X.head())

# Create training and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# Create DataFrame version with feature names for SHAP
X_test_df = pd.DataFrame(X_test_s, columns=X.columns.tolist())
feature_names = X.columns.tolist()

print(f"Training set: {X_train_s.shape}, Test set: {X_test_s.shape}")
print(f"Classes: {dict(zip(data.target_names, [sum(y_test==0), sum(y_test==1)]))}")

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Training set: (455, 30), Test set: (114, 30)
Classes: {np.str_('malignant'): 43, np.str_('benign'): 71}


Complete the coding exercises below.

## LIME

- **Task (0.5 points):** Select a model (e.g. random forest, gradient boosting, neural network) to address the given task and fit it on the dataset.
- **Task (0.5 points):** Run LIME explainer for the trained model.
- **Task (1 point):** Run local explanations for 4 test samples of your choice from different classes.
- **Task (1 point):** Barplot of the top 5 most influential and top 5 least influential global features created by averaging the feature importances from 100 randomly chosen test samples. Looking at the bar plot of the least influential global features, do you think these features are genuinely uninformative for cancer diagnosis, or could the model be underutilising potentially relevant signals?
- **Task (1 point):** Plot a global overview of the most important features for the model's predictions for the same 100 randomly chosen test samples in the previous task using SP-LIME.
- **Task (1 point):** Create a heatmap where each row is one of the 100 samples and each column is a feature, with cell colour showing the LIME importance weight. This lets you see at a glance which features drive predictions consistently across samples and which vary. Does the heatmap reveal consistent or varying importance of features across samples?

### Coding Tasks

**Task (0.5 points):** Select a model (e.g. random forest, gradient boosting, neural network) to address the given task and fit it on the dataset.

In [5]:
# Your solution here...
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

class MyNeurons(nn.Module):

  def __init__(self, num_inputs, num_hidden, num_outputs):
    super().__init__()
    # Some init for my module
    self.linear1 = nn.Linear(num_inputs, num_hidden)
    self.act_fn = nn.ReLU()
    self.linear2 = nn.Linear(num_hidden, num_outputs)


  def forward(self, x):
    # Function for performing the calculation of the module.
    x = self.linear1(x)
    x = self.act_fn(x)
    x = self.linear2(x)
    return x

class MyDataset(torch.utils.data.Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x.T,dtype=torch.float32)
    self.y = torch.tensor(y,dtype=torch.float32)
    self.length = self.x.shape[0]

  def __len__(self):
      return self.length

  def __getitem__(self,idx):
      return self.x[idx],self.y

model = MyNeurons(30, 10, 1)
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()

dataset = MyDataset(X_train_s,y_train)
dataLoader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

def tarin_model(model, optimizer, dataLoader, loss_fn, num_epochs):
  model.train()

  for epoch in range(num_epochs):
    for x,y in dataLoader:
      data_in = x.to(device)
      target = y.to(device)

      pred = model(data_in)
      pred = pred.squeeze(1)
      loss = loss_fn(pred, target)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  return model

tarin_model(model, optimizer, dataLoader, loss_fn, 10)

trainDataset = MyDataset(X_train_s,y_train)
dataLoadertest = torch.utils.data.DataLoader(trainDataset, batch_size=32, shuffle=False)

def eval_Model(model, testDateLoader):
  model.eval()
  true_preds, num_preds = 0.,0.
  with torch.no_grad():
    predictions = []
    for x,y in testDateLoader:
      data_in = x.to(device)
      data_out = model(data_in)
      data_out = data_out.squeeze(1)
      data_out = torch.sigmoid(data_out)
      pred_labels = (data_out >= 0.5).long()

      true_preds += (pred_labels == y.to(device)).sum()
      num_preds += y.shape[0]
      predictions.append(data_out.cpu().numpy())

  acc = true_preds / num_preds
  return acc, predictions

acc, predictions = eval_Model(model, dataLoadertest)
print(f"Accuracy: {acc}")


ValueError: Target size (torch.Size([32, 455])) must be the same as input size (torch.Size([32]))

**Task (0.5 points):** Run LIME explainer for the trained model.

In [ ]:
# Your solution here...

**Task (1 point):** Run local explanations for 4 test samples of your choice from different classes.

In [ ]:
# Your solution here...

**Task (1 point):** Barplot of the top 5 most influential and top 5 least influential global features created by averaging the feature importances from 100 randomly chosen test samples. Looking at the bar plot of the least influential global features, do you think these features are genuinely uninformative for cancer diagnosis, or could the model be underutilising potentially relevant signals?

In [ ]:
# Your solution here...

**Task (1 point):** Plot a global overview of the most important features for the model's predictions for the same 100 randomly chosen test samples in the previous task using SP-LIME.

In [ ]:
# Your solution here...

**Task (1 point):** Create a heatmap where each row is one of the 100 samples and each column is a feature, with cell colour showing the LIME importance weight. This lets you see at a glance which features drive predictions consistently across samples and which vary. Does the heatmap reveal consistent or varying importance of features across samples?

In [ ]:
# Your solution here...

## SHAP

- **Task (1 point):** Apply SHAP explainer on the trained model. Plot a force plot for the same 4 test samples you had selected for LIME.
- **Task (1 point):** Visualize how each feature contributes to the predicted probability of the selected 4 test samples using the waterfall plot. Compare the results to those from LIME for the same samples. What do you observe?
- **Task (0.5 points):** Plot a summary plot (global overview) of how each feature affects the model's prediction for the complete test set.
- **Task (1 point):** Create a SHAP explainer for the same 100 random test samples you used in the LIME task. Plot a global overview of how each feature affects the model prediction. How does this compare to the global explanations from SP-LIME?
- **Task (0.5 points):** Make a data frame that contains the top 5 most influential features for all test samples.
- **Task (1 point):** Plot a dependence plot for two of the most important features: `worst concave points` and `worst area`.

**Task (1 point):** Apply SHAP explainer on the trained model. Plot a force plot for the same 4 test samples you had selected for LIME.

In [ ]:
# Your solution here...

**Task (1 point):** Visualize how each feature contributes to the predicted probability of the selected 4 test samples using the waterfall plot. Compare the results to those from LIME for the same samples. What do you observe?

In [ ]:
# Your solution here...

**Task (0.5 points):** Plot a summary plot of how each feature affects the model's prediction for the complete test set.

In [ ]:
# Your solution here...

**Task (1 point):** Create a SHAP explainer for the same 100 random test samples you used in the LIME task. Plot a global overview of how each feature affects the model prediction. How does this compare to the global explanations from SP-LIME?

In [ ]:
# Your solution here...

**Task (0.5 points):** Make a data frame that contains the top 5 most influential features for all test samples.

In [ ]:
# Your solution here...

**Task (1 point):** Plot a dependence plot for two of the most important features: `worst concave points` and `worst area`.

In [ ]:
# Your solution here...